# Content Analysis — Nigeria

**Norman Lear Center × Gates Foundation — Manfluencer project**

Generates **coding units** from creator content for qualitative theme/sentiment coding. Output schema matches the Kibe/Jagero reference:

| Segment ID | Influencer | Platform | Content Type | Theme(s) | Context (NOT CODED) | Verbatim Text (CODE THIS) |

## Sources

- **Banky Wellington** — 6 MENtality podcast transcripts → ~40 coding units per video via `gpt-4o`
- **Deyemi Okanlawon, Shola, Agba John Doe, Wizarab** — scope-relevant X tweets → 1 coding unit per tweet via `gpt-4o-mini`

## Outputs

`Nigeria/Content Analysis/Content - Raw/<Creator>/<Creator>_X_Tweets.xlsx` — coded tweets, 1 row per tweet.


## Setup

In [ ]:
from __future__ import annotations
import os, json, asyncio, re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as atqdm

ROOT = Path.cwd().parents[1] if Path.cwd().name == "Notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing"
print("ROOT:", ROOT)


## Stage 1 — Code X tweets into coding units (gpt-4o-mini)

In [ ]:
TWEETS_DIR = ROOT / "Nigeria" / "Scraped Tweets"
OUT_DIR    = ROOT / "Nigeria" / "Content Analysis" / "Content - Raw"
TEMP_DIR   = ROOT / "temp" / "content_analysis_tweets"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

CREATORS = [
    {"name": "Deyemi Okanlawon", "orientation": "Progressive", "id_prefix": "DEY"},
    {"name": "Agba John Doe",    "orientation": "Regressive",  "id_prefix": "AGB"},
    {"name": "Shola",            "orientation": "Regressive",  "id_prefix": "SHO"},
    {"name": "Wizarab",          "orientation": "Regressive",  "id_prefix": "WIZ"},
]

LLM_MODEL = "gpt-4o-mini"
BATCH_SIZE = 15
CONCURRENCY = 12


### LLM prompt — theme tagging + concise context

For each tweet, gpt-4o-mini returns 1-3 themes from the canonical masculinity/gender taxonomy plus a one-sentence context note for the human coder.


In [ ]:
CANONICAL_THEMES = [
    "marriage_relationships", "infidelity_cheating", "polygamy_scarcity",
    "female_submission", "men_as_prize", "marriage_market_logic",
    "dating_standards", "sexual_double_standards",
    "fatherhood_parenting", "raising_boys", "male_role_models",
    "men_money_provider", "wealth_status",
    "male_emotional_life", "vulnerability_mental_health", "male_friendship",
    "rape_sexual_violence", "false_accusations", "boy_child_male_victim",
    "gender_debate_feminism", "anti_women", "anti_feminism", "not_all_men_deflection",
    "religion_faith_framing", "biblical_marriage_framing",
    "self_promotion_book_show", "humor_meme_pidgin",
    "other_off_scope",
]

SYSTEM_PROMPT = """You are a research coder for a study of Nigerian masculinity influencers (regressive vs progressive). Each input is one X / Twitter post by one of: Deyemi Okanlawon (progressive), Agba John Doe (regressive marriage advice), Shola (regressive scarcity/hierarchy), Wizarab (regressive anti-women).

Your job per tweet:
1. Pick 1-3 THEMES from this exact list (semicolon-separated):
   {themes}
2. Write ONE concise context sentence for the human coder to understand the tweet (NOT for coding — comprehension only). Include any reference (case names, platforms, slang) the coder might miss. Max 25 words.

Return JSON only:
{{"results": [{{"id": <int>, "themes": "theme1; theme2", "context": "..."}}]}}"""

SYSTEM_PROMPT = SYSTEM_PROMPT.format(themes=", ".join(CANONICAL_THEMES))

def build_user_prompt(creator_name, batch):
    lines = [f"Creator: {creator_name}", "", "Tweets:"]
    for i, text in batch:
        lines.append(f"[{i}] {str(text).replace(chr(10), ' ')[:600]}")
    return "\n".join(lines)


In [ ]:
async def code_batch(client, sem, creator_name, idx_to_text):
    items = list(idx_to_text.items())
    async with sem:
        for attempt in range(4):
            try:
                resp = await client.chat.completions.create(
                    model=LLM_MODEL, temperature=0,
                    response_format={"type": "json_object"},
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": build_user_prompt(creator_name, items)},
                    ],
                )
                data = json.loads(resp.choices[0].message.content)
                out = {}
                for r in data.get("results", []):
                    rid = r.get("id")
                    if rid in idx_to_text:
                        out[rid] = (str(r.get("themes", "")), str(r.get("context", "")))
                return out
            except Exception as e:
                if attempt == 3:
                    print(f"  ! batch failed: {e}")
                    return {}
                await asyncio.sleep(2 ** attempt)


async def code_creator(creator):
    in_path = TWEETS_DIR / f"{creator['name']}_all_tweets.xlsx"
    cache_path = TEMP_DIR / f"{creator['name'].replace(' ','_')}.cache.json"
    df = pd.read_excel(in_path)
    print(f"\n=== {creator['name']} ({len(df)} tweets) ===")

    cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}

    pending = {i: row["text"] for i, row in df.iterrows() if str(i) not in cache}
    print(f"  cached: {len(cache)}, pending: {len(pending)}")

    if pending:
        client = AsyncOpenAI()
        sem = asyncio.Semaphore(CONCURRENCY)
        coros = []
        items = list(pending.items())
        for start in range(0, len(items), BATCH_SIZE):
            chunk = dict(items[start:start + BATCH_SIZE])
            coros.append(code_batch(client, sem, creator["name"], chunk))
        results = await atqdm.gather(*coros, desc=f"{creator['name']} batches")
        for batch_out in results:
            for rid, (themes, ctx) in batch_out.items():
                cache[str(rid)] = {"themes": themes, "context": ctx}
        cache_path.write_text(json.dumps(cache, indent=1))

    # Build the coding-unit dataframe
    rows = []
    for i, row in df.iterrows():
        meta = cache.get(str(i), {"themes": "other_off_scope", "context": ""})
        rows.append({
            "Segment ID": f"{creator['id_prefix']}_{i+1:03d}",
            "Influencer": creator["name"],
            "Platform": "X",
            "Content Type": "Tweet",
            "Theme(s)": meta["themes"],
            "Context (NOT CODED - comprehension only)": meta["context"],
            "Verbatim Text (CODE THIS)": row["text"],
        })
    out_df = pd.DataFrame(rows)

    out_dir = OUT_DIR / creator["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{creator['name']}_X_Tweets.xlsx"
    out_df.to_excel(out_path, index=False)
    print(f"  → {out_path.relative_to(ROOT)}: {len(out_df)} coded tweets")
    return out_path


async def main():
    paths = []
    for c in CREATORS:
        try:
            paths.append(await code_creator(c))
        except Exception as e:
            print(f"  ✗ failed for {c['name']}: {e}")
    print("\n=== ALL DONE ===")
    for p in paths:
        df = pd.read_excel(p)
        print(f"  {p.name}: {len(df)} rows")

await main()


## Stage 2 — Banky transcript coding units (placeholder)

In [ ]:
# To be added once transcripts finish — splits each MENtality transcript into ~40 coding units via gpt-4o.
print('Pending — Banky transcripts still being generated by Data Acquisition Pipeline.')
